In [1]:
import sys
sys.path.append("..")

In [2]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import ipywidgets as widgets
from ipywidgets import interact
from scipy.optimize import curve_fit
from pipeline.ingesta import cargar_indices_desde_bd
from utils.aplicar_whittaker import aplicar_whittaker_series

from pipeline.modulo_fenologico import segmentar_ciclos, detectar_sos
# Ajusta esta ruta al módulo real donde viven las funciones que pegaste
from pipeline.modulo_predictivo import (
    ajustar_curva_doble_logistica,
    extender_serie_con_curva_parametrica,
)

FECHA_INICIO = "2021-01-01"
FECHA_FIN = "2025-12-31"
DISTANCIA_MIN_DIAS = 90
PROMINENCIA_MIN = 0.15
FACTOR_SOS = 0.2
EOS_DIAS = 120  # SOS + 120, tu convención ya establecida

2026-07-07 23:41:13.161 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-07-07 23:41:13.163 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-07-07 23:41:13.164 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-07-07 23:41:13.165 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-07-07 23:41:13.166 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-07-07 23:41:13.167 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-07-07 23:41:13.168 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-07-07 23:41:13.169 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-07-

In [3]:
dfs_raw = cargar_indices_desde_bd(FECHA_INICIO, FECHA_FIN)
dfs_suave = aplicar_whittaker_series(dfs_raw, lambda_param=10000.0, orden=2)

print("Índices disponibles:", list(dfs_raw.keys()))
print("Parcelas:", dfs_raw["EVI"].columns.tolist())

✅  Índices cargados desde BD: 1826 fechas × 9 parcelas (2021-01-01 → 2025-12-31).
📈 Suavizando serie temporal para: EVI...
📈 Suavizando serie temporal para: LSWI...
Índices disponibles: ['EVI', 'LSWI']
Parcelas: ['id_0', 'id_1', 'id_2', 'id_3', 'id_4', 'id_5', 'id_6', 'id_7', 'id_8']


In [4]:
def obtener_ciclos_con_sos(parcela: str, indice_nombre: str = "EVI") -> list[dict]:
    """
    Replica la lógica de segmentación + detección de SOS de
    graficar_whittaker_sos, pero devuelve los resultados como estructura
    de datos en vez de graficarlos directamente. Cada ciclo detectado
    incluye sus límites (valles) y su fecha SOS si se pudo detectar.
    """
    serie = dfs_suave[indice_nombre][parcela]
    segmentos = segmentar_ciclos(serie, DISTANCIA_MIN_DIAS, PROMINENCIA_MIN)

    ciclos = []
    for inicio, fin in segmentos:
        es_ultimo = (inicio, fin) == segmentos[-1]
        mask = (serie.index >= inicio) & (serie.index <= fin) if not es_ultimo else (serie.index >= inicio)
        df_segmento = serie.loc[mask]

        if df_segmento.empty:
            continue

        resultado_sos = detectar_sos(
            serie=df_segmento.values,
            fechas=df_segmento.index,
            factor=FACTOR_SOS,
            metodo="seasonal_amplitude",
            ventana_busqueda=(inicio, serie.index.max()) if es_ultimo else (inicio, fin),
        )
        fecha_sos = resultado_sos.get("sos_fecha")
        if fecha_sos is None:
            continue

        ciclos.append({
            "inicio_valle": inicio,
            "fin_valle": fin,
            "fecha_sos": pd.Timestamp(fecha_sos),
            "fecha_eos": pd.Timestamp(fecha_sos) + pd.Timedelta(days=EOS_DIAS),
        })
    return ciclos

In [5]:
def simular_extrapolacion_ventana(
    parcela: str,
    indice_nombre: str,
    ciclo: dict,
    ventana: str,  # "T1", "T2", "T3"
) -> dict:
    """
    Trunca artificialmente la serie suavizada de un ciclo en el día
    correspondiente a la ventana (30/60/90 post-SOS), extrapola hasta EOS,
    y conserva también la serie real completa del ciclo para poder
    comparar visualmente la extrapolación contra lo que de verdad ocurrió.
    """
    dias_ventana = {"T1": 30, "T2": 60, "T3": 90}[ventana]
    fecha_sos = ciclo["fecha_sos"]
    fecha_eos = ciclo["fecha_eos"]
    fecha_corte = fecha_sos + pd.Timedelta(days=dias_ventana)

    serie_completa = dfs_suave[indice_nombre][parcela]
    serie_ciclo_real = serie_completa.loc[fecha_sos:fecha_eos]

    # "Lo que se hubiera conocido" hasta la ventana T simulada
    serie_truncada = serie_ciclo_real.loc[:fecha_corte]

    if fecha_corte not in serie_ciclo_real.index:
        return None  # el ciclo aún no llega a esta ventana, no se puede simular

    serie_extendida, params = extender_serie_con_curva_parametrica(
        serie_suavizada=serie_truncada,
        fecha_sos=fecha_sos,
        fecha_fin_extension=fecha_eos,
    )

    # Solo la porción extrapolada (posterior al corte), para graficarla aparte
    serie_solo_extrapolada = serie_extendida.loc[fecha_corte:]

    return {
        "fecha_corte": fecha_corte,
        "fecha_sos": fecha_sos,
        "fecha_eos": fecha_eos,
        "serie_truncada": serie_truncada,
        "serie_extrapolada": serie_solo_extrapolada,
        "serie_real_completa": serie_ciclo_real,
        "params": params,
    }

In [6]:
def graficar_validacion_extrapolacion(parcela: str, indice_nombre: str, ventana: str):
    ciclos = obtener_ciclos_con_sos(parcela, indice_nombre)
    if not ciclos:
        print("No se detectaron ciclos con SOS válido para esta parcela.")
        return

    opciones_ciclo = {
        f"SOS {c['fecha_sos'].date()} → EOS {c['fecha_eos'].date()}": c
        for c in ciclos
    }

    def actualizar(ciclo_label):
        ciclo = opciones_ciclo[ciclo_label]
        resultado = simular_extrapolacion_ventana(parcela, indice_nombre, ciclo, ventana)

        if resultado is None:
            print(f"Este ciclo no alcanza la ventana {ventana} (termina antes del día correspondiente).")
            return

        fig = go.Figure()

        # A. Serie real completa del ciclo (referencia, gris tenue de fondo)
        fig.add_trace(go.Scatter(
            x=resultado["serie_real_completa"].index,
            y=resultado["serie_real_completa"].values,
            mode="lines",
            line=dict(color="lightgray", width=2),
            name="Serie real completa (ground truth histórico)",
        ))

        # B. Porción "conocida" hasta la ventana T (sólido azul)
        fig.add_trace(go.Scatter(
            x=resultado["serie_truncada"].index,
            y=resultado["serie_truncada"].values,
            mode="lines",
            line=dict(color="#1f77b4", width=2.5),
            name=f"Suavizada observada hasta {ventana}",
        ))

        # C. Extrapolación: discontinua, empieza exactamente en el punto de corte
        serie_extrap = resultado["serie_extrapolada"]
        fig.add_trace(go.Scatter(
            x=serie_extrap.index,
            y=serie_extrap.values,
            mode="lines",
            line=dict(color="#d62728", width=2.5, dash="dash"),
            name="Extrapolación (curva doble logística)",
        ))

        # D. Líneas de referencia: SOS, corte de ventana, EOS
        fig.add_vline(x=ciclo["fecha_sos"], line_width=1.5, line_dash="dot", line_color="green")
        fig.add_annotation(x=ciclo["fecha_sos"], y=1, yref="paper", text="SOS", showarrow=False,
                            textangle=-90, xanchor="right", yanchor="top", font=dict(color="green", size=9))

        fig.add_vline(x=resultado["fecha_corte"], line_width=1.5, line_dash="dot", line_color="#d62728")
        fig.add_annotation(x=resultado["fecha_corte"], y=1, yref="paper", text=f"Corte {ventana}",
                            showarrow=False, textangle=-90, xanchor="right", yanchor="top",
                            font=dict(color="#d62728", size=9))

        fig.add_vline(x=ciclo["fecha_eos"], line_width=1.5, line_dash="dot", line_color="gray")
        fig.add_annotation(x=ciclo["fecha_eos"], y=1, yref="paper", text="EOS (SOS+120)",
                            showarrow=False, textangle=-90, xanchor="right", yanchor="top",
                            font=dict(color="gray", size=9))

        # E. Métrica de calidad del ajuste en el título
        params = resultado["params"]
        r2_txt = f"R²={params['r2']:.3f}" if params is not None else "ajuste no confiable (sin extrapolar)"

        fig.update_layout(
            title=dict(
                text=f"Validación extrapolación: {indice_nombre} en {parcela} — Ventana {ventana} ({r2_txt})",
                font=dict(size=14, weight="bold"),
            ),
            xaxis_title="Fecha",
            yaxis_title=indice_nombre,
            template="plotly_white",
            hovermode="x unified",
            legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
            width=1100,
            height=550,
            margin=dict(t=90, b=40, l=50, r=40),
        )
        fig.show()

    interact(
        actualizar,
        ciclo_label=widgets.Dropdown(options=list(opciones_ciclo.keys()), description="Ciclo:"),
    )

In [ ]:
graficar_validacion_extrapolacion(
    parcela="id_3",       # ajusta al id real de tu parcela
    indice_nombre="EVI",  # o "LSWI"
    ventana="T2",         # "T1", "T2" o "T3"
)

interactive(children=(Dropdown(description='Ciclo:', options=('SOS 2022-08-17 → EOS 2022-12-15', 'SOS 2023-04-…

In [10]:
def calcular_fecha_fin_extension_segura(
    ciclo: dict,
    ciclos_ordenados: list[dict],
    margen_dias: int = 10,
) -> pd.Timestamp:
    """
    Determina la fecha límite de extrapolación para un ciclo, capando
    SOS+120 si el siguiente ciclo detectado arranca antes de esa fecha.
    Evita que la curva doble logística intente modelar el arranque del
    siguiente ciclo, para el cual no tiene forma funcional adecuada.
    """
    idx = ciclos_ordenados.index(ciclo)
    eos_teorico = ciclo["fecha_eos"]  # SOS + 120

    if idx + 1 < len(ciclos_ordenados):
        siguiente_sos = ciclos_ordenados[idx + 1]["fecha_sos"]
        limite_por_contaminacion = siguiente_sos - pd.Timedelta(days=margen_dias)
        return min(eos_teorico, limite_por_contaminacion)

    return eos_teorico

In [11]:
from pipeline.modulo_predictivo import _doble_logistica
def evaluar_plausibilidad_extrapolacion(
    serie_truncada: pd.Series,
    params: dict,
    fecha_sos: pd.Timestamp,
    dias_ventana_tendencia: int = 7,
) -> dict:
    """
    Compara la tendencia local observada (últimos N días antes del corte)
    contra la tendencia que implica la curva ajustada en ese mismo punto.
    Una discrepancia fuerte de signo indica que el modelo unimodal no
    captura la dinámica real (ej. inicio de un nuevo ciclo bleeding-in),
    aunque el R² del ajuste sobre los datos observados sea alto.
    """
    y_recientes = serie_truncada.iloc[-dias_ventana_tendencia:].to_numpy()
    x_recientes = np.arange(len(y_recientes), dtype=float)
    pendiente_observada = np.polyfit(x_recientes, y_recientes, 1)[0]

    ultimo_dia = (serie_truncada.index[-1] - fecha_sos).days
    delta = 1.0
    y0 = _doble_logistica(ultimo_dia, **{k: params[k] for k in ["vmin","vmax","S","mS","A","mA"]})
    y1 = _doble_logistica(ultimo_dia + delta, **{k: params[k] for k in ["vmin","vmax","S","mS","A","mA"]})
    pendiente_modelo = (y1 - y0) / delta

    signos_coinciden = np.sign(pendiente_observada) == np.sign(pendiente_modelo)
    magnitud_razonable = abs(pendiente_modelo) < 3 * abs(pendiente_observada) + 1e-3

    return {
        "pendiente_observada": pendiente_observada,
        "pendiente_modelo": pendiente_modelo,
        "plausible": bool(signos_coinciden and magnitud_razonable),
    }

In [12]:
def validar_extrapolaciones_historicas(
    parcelas: list[str],
    indice_nombre: str,
    ventanas: list[str] = ["T1", "T2", "T3"],
) -> pd.DataFrame:
    """
    Recorre todos los ciclos históricos de las parcelas dadas y calcula
    el error de la extrapolación contra la serie real observada (que en
    modo histórico ya se conoce), para cada ventana T. Permite detectar
    sistemáticamente los casos donde el modelo doble-logístico falla
    (ej. contaminación por ciclo siguiente), en vez de depender de
    inspección visual caso por caso.
    """
    filas = []
    for parcela in parcelas:
        ciclos = obtener_ciclos_con_sos(parcela, indice_nombre)
        ciclos_ordenados = sorted(ciclos, key=lambda c: c["fecha_sos"])

        for ciclo in ciclos_ordenados:
            fecha_eos_segura = calcular_fecha_fin_extension_segura(ciclo, ciclos_ordenados)

            for ventana in ventanas:
                resultado = simular_extrapolacion_ventana(parcela, indice_nombre, ciclo, ventana)
                if resultado is None or resultado["params"] is None:
                    continue

                plausibilidad = evaluar_plausibilidad_extrapolacion(
                    resultado["serie_truncada"], resultado["params"], ciclo["fecha_sos"]
                )

                # Comparar extrapolación vs realidad SOLO dentro de la ventana segura
                real_post_corte = resultado["serie_real_completa"].loc[
                    resultado["fecha_corte"]:fecha_eos_segura
                ]
                extrap_post_corte = resultado["serie_extrapolada"].loc[
                    :fecha_eos_segura
                ].reindex(real_post_corte.index)

                error = (extrap_post_corte - real_post_corte).dropna()
                rmse = np.sqrt((error ** 2).mean()) if len(error) > 0 else np.nan

                filas.append({
                    "parcela": parcela,
                    "sos": ciclo["fecha_sos"],
                    "ventana": ventana,
                    "r2_ajuste": resultado["params"]["r2"],
                    "rmse_extrapolacion": rmse,
                    "plausible": plausibilidad["plausible"],
                })

    return pd.DataFrame(filas)

In [13]:
df_validacion = validar_extrapolaciones_historicas(
    parcelas=dfs_suave["EVI"].columns.tolist(),
    indice_nombre="EVI",
)
df_validacion.sort_values("rmse_extrapolacion", ascending=False).head(15)

,parcela,sos,ventana,r2_ajuste,rmse_extrapolacion,plausible
162,id_8,2025-05-08,T2,0.999587,0.677324,False
64,id_2,2022-10-04,T2,0.999858,0.672946,True
85,id_3,2024-05-26,T2,1.000000,0.382630,True
26,id_0,2025-04-07,T3,0.998654,0.333292,False
59,id_1,2025-03-25,T3,0.999590,0.310025,True
118,id_6,2022-06-27,T2,0.999997,0.224478,True
21,id_0,2024-09-26,T1,0.999999,0.208955,True
36,id_1,2022-05-28,T1,0.999999,0.205551,True
12,id_0,2023-03-02,T1,1.000000,0.204612,True
112,id_5,2024-06-20,T2,0.999999,0.203384,True
